# Python Object-Oriented Programming Guide

A practical, runnable guide to Python OOP. Work through the notebook from top to bottom: explanations introduce each idea, and code cells demonstrate it.

Topics include classes and objects, attributes, methods, the four OOP pillars, inheritance and MRO, composition, properties, special methods, equality, and copying.

## 1. Classes and objects

A **class** defines the data and behavior that its objects can have. An **object** (or instance) is a value created from that class. `__init__` initializes a new instance, and `self` refers to that particular instance. Python passes `self` automatically when a bound instance method is called.

In [1]:
class Employee:
    def __init__(self, name, salary):
        self.name = name          # Store data on this particular object.
        self.salary = salary

    def describe(self):
        return f"{self.name} earns ${self.salary:,} per year."

employee_one = Employee("Rohit", 60000)
employee_two = Employee("Akshay", 72000)
print(employee_one.describe())
print(employee_two.describe())
print(employee_one is employee_two)  # Two separate objects.

Rohit earns $60,000 per year.
Akshay earns $72,000 per year.
False


`employee_one.describe()` is convenient syntax for `Employee.describe(employee_one)`. This is why the method definition includes `self`, even though callers normally do not pass it explicitly.

## 2. Instance attributes and class attributes

An **instance attribute** belongs to one object. A **class attribute** is defined on the class and is shared unless an instance shadows it. For ordinary attribute lookup, Python checks the instance before looking up the class. Assigning through an instance creates or updates an instance attribute; it does not change the class attribute.

In [2]:
class TeamMember:
    company = "Northstar Labs"  # Shared class attribute.

    def __init__(self, name):
        self.name = name          # Separate instance attribute.

member_one = TeamMember("Amit")
member_two = TeamMember("Priya")
member_one.company = "Independent"  # Shadows the class attribute for this object.

print(member_one.company)
print(member_two.company)
print(TeamMember.company)

Independent
Northstar Labs
Northstar Labs


### The mutable class-attribute pitfall

A mutable class attribute, such as a list, is one shared object. Calling `.append()` through `self` mutates that shared list; it does not create an instance attribute. Put per-object mutable data in `__init__` instead.

In [3]:
class SkillProfile:
    def __init__(self, name):
        self.name = name
        self.skills = []  # Each object receives a distinct list.

    def add_skill(self, skill):
        self.skills.append(skill)

profile_one = SkillProfile("Amit")
profile_two = SkillProfile("Priya")
profile_one.add_skill("Python")
print(profile_one.skills)
print(profile_two.skills)

['Python']
[]


## 3. Instance, class, and static methods

- An **instance method** receives `self` and usually works with one object's state.
- A **class method** uses `@classmethod` and receives `cls`; it works with class-level state and is often used as an alternative constructor.
- A **static method** uses `@staticmethod` and receives no automatic `self` or `cls`; it is a related utility function.

In [4]:
class Temperature:
    scale = "Celsius"

    def __init__(self, degrees):
        self.degrees = degrees

    def as_text(self):
        return f"{self.degrees} degrees {self.scale}"  # Uses this instance.

    @classmethod
    def from_fahrenheit(cls, value):
        celsius = (value - 32) * 5 / 9
        return cls(celsius)  # cls keeps this constructor friendly to subclasses.

    @staticmethod
    def is_above_absolute_zero(degrees):
        return degrees >= -273.15

room = Temperature(21)
freezing = Temperature.from_fahrenheit(32)
print(room.as_text())
print(freezing.as_text())
print(Temperature.is_above_absolute_zero(-300))

21 degrees Celsius
0.0 degrees Celsius
False


## 4. Encapsulation and properties

Encapsulation groups state and behavior and offers a clear way to work with that state. Python does not enforce strict private attributes: `_value` signals an internal implementation detail by convention, while a double underscore triggers name mangling to reduce accidental name collisions. A property can provide attribute-style access while validating updates.

In [5]:
class BankAccount:
    def __init__(self, opening_balance):
        if opening_balance < 0:
            raise ValueError("Opening balance cannot be negative")
        self.__balance = opening_balance  # Name-mangled, not strictly private.

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self.__balance += amount

    @property
    def balance(self):
        return self.__balance  # Read access, but no public direct assignment.

account = BankAccount(1000)
account.deposit(250)
print(account.balance)

try:
    account.deposit(-10)
except ValueError as error:
    print(f"Rejected deposit: {error}")

1250
Rejected deposit: Deposit must be positive


### A validating property setter

`@property` defines a getter, and `@price.setter` defines assignment behavior. The object can preserve a simple `product.price` interface while guarding its internal state. The invalid example below is caught so later notebook cells remain runnable.

In [6]:
class Product:
    def __init__(self, price):
        self.price = price  # Goes through the setter and validation.

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError("Price cannot be negative")
        self._price = value

product = Product(100)
product.price = 125
print(product.price)

try:
    product.price = -50
except ValueError as error:
    print(f"Invalid update: {error}")

print(product.price)  # The rejected update left the original price unchanged.

125
Invalid update: Price cannot be negative
125


## 5. Inheritance and method overriding

Inheritance lets a child class reuse behavior from a parent class. A child can **override** an inherited method with a specialized implementation. `pass` is a valid placeholder when a class or method intentionally has no body yet.

In [7]:
class Animal:
    def speak(self):
        return "Some animal sound"

class Dog(Animal):
    def speak(self):  # Overrides Animal.speak.
        return "Bark"

class Cat(Animal):
    pass  # Inherits Animal.speak unchanged.

print(Dog().speak())
print(Cat().speak())

Bark
Some animal sound


## 6. Polymorphism and duck typing

Polymorphism means the same operation can work with different object types and produce type-specific behavior. Python often uses **duck typing**: code can accept any object that provides the operation it needs, without requiring a particular parent class.

In [8]:
class EmailSender:
    def send(self, message):
        return f"Email sent: {message}"

class TextSender:
    def send(self, message):
        return f"Text sent: {message}"

class ConsoleSender:
    def send(self, message):
        return f"Console message: {message}"

def notify(sender, message):
    # This function only requires the object to provide send().
    return sender.send(message)

for sender in (EmailSender(), TextSender(), ConsoleSender()):
    print(notify(sender, "Build complete"))

Email sent: Build complete
Text sent: Build complete
Console message: Build complete


## 7. Abstraction with abstract base classes

An abstract base class (ABC) defines a required interface. A class with an unimplemented `@abstractmethod` cannot be instantiated. Every concrete subclass must implement the required method before it can be created.

In [9]:
from abc import ABC, abstractmethod

class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount):
        """Process a payment and return a receipt description."""
        raise NotImplementedError

class CardPayment(PaymentMethod):
    def pay(self, amount):
        return f"Paid ${amount:.2f} by card"

class WalletPayment(PaymentMethod):
    def pay(self, amount):
        return f"Paid ${amount:.2f} by wallet"

for method in (CardPayment(), WalletPayment()):
    print(method.pay(25))

# Creating PaymentMethod() directly would raise TypeError because pay() is abstract.

Paid $25.00 by card
Paid $25.00 by wallet


The ABC enforces the interface at instantiation time. This is different from duck typing, which generally relies on the operation being available when the code calls it. Both approaches can be useful; choose the simplest contract that fits the design.

## 8. `super()` and constructor inheritance

`super()` delegates to the next class in the method resolution order (MRO). It is useful for reusing parent behavior without hardcoding the parent class. If a child does not define `__init__`, it inherits the parent's initializer. If it defines its own, the parent initializer runs only when explicitly called, commonly with `super().__init__(...)`.

In [10]:
class NamedAnimal:
    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"Animal: {self.name}"

class GuideDog(NamedAnimal):
    def __init__(self, name, certification):
        super().__init__(name)  # Reuse parent initialization.
        self.certification = certification

    def describe(self):
        parent_description = super().describe()  # Reuse parent behavior.
        return f"{parent_description}; certified: {self.certification}"

dog = GuideDog("Buddy", "Level 2")
print(dog.describe())

Animal: Buddy; certified: Level 2


In [11]:
class ParentInitializer:
    def __init__(self):
        self.parent_ready = True

class InheritsInitializer(ParentInitializer):
    pass

class CallsParentInitializer(ParentInitializer):
    def __init__(self):
        super().__init__()
        self.child_ready = True

inherited = InheritsInitializer()
explicit = CallsParentInitializer()
print(inherited.parent_ready)
print(explicit.parent_ready, explicit.child_ready)

True
True True


## 9. Multiple inheritance and MRO

Python's MRO is the ordered path used to find methods in an inheritance graph. `super()` follows this order; it does not simply mean 'call my direct parent'. The cooperative example below has each method call `super()` so each class in the MRO participates once.

In [12]:
class Root:
    def show(self):
        print("Root")

class Left(Root):
    def show(self):
        print("Left")
        super().show()

class Right(Root):
    def show(self):
        print("Right")
        super().show()

class Combined(Left, Right):
    def show(self):
        print("Combined")
        super().show()

print([class_type.__name__ for class_type in Combined.mro()])
Combined().show()  # Output order follows the MRO.

['Combined', 'Left', 'Right', 'Root', 'object']
Combined
Left
Right
Root


## 10. Composition versus inheritance

Inheritance models an **is-a** relationship (a `Dog` is an `Animal`). Composition models a **has-a** relationship (a `Car` has an `Engine`). Composition often makes it easier to swap one component for another without changing the containing class.

In [13]:
class ElectricMotor:
    def start(self):
        return "Electric motor started"

class GasEngine:
    def start(self):
        return "Gas engine started"

class CarWithEngine:
    def __init__(self, engine):
        self.engine = engine  # The car contains an engine object.

    def start(self):
        return f"{self.engine.start()}; car ready"

electric_car = CarWithEngine(ElectricMotor())
gas_car = CarWithEngine(GasEngine())
print(electric_car.start())
print(gas_car.start())

Electric motor started; car ready
Gas engine started; car ready


## 11. Class variables and object counting

A class variable can hold information shared by instances, such as a count. Use the class to update and access shared state explicitly. Be aware that assigning `some_instance.count = ...` creates an instance attribute that shadows the class value for that one object.

In [14]:
class Visitor:
    count = 0

    def __init__(self, name):
        self.name = name
        Visitor.count += 1

visitor_one = Visitor("Amit")
visitor_two = Visitor("Priya")
print(Visitor.count)

visitor_one.count = 99  # Shadows the class attribute on visitor_one only.
print(visitor_one.count)
print(visitor_two.count)
print(Visitor.count)

2
99
2
2


## 12. String representations: `__str__` and `__repr__`

`__str__` is intended to be readable for people and is used by `str()` and `print()`. `__repr__` is intended to help developers inspect or recreate a value. If `__str__` is missing, Python can fall back to `__repr__`. Both methods should return strings.

In [15]:
class BookDescription:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages

    def __str__(self):
        return f"{self.title} ({self.pages} pages)"

    def __repr__(self):
        return f"BookDescription({self.title!r}, {self.pages!r})"

book = BookDescription("Python Patterns", 240)
print(str(book))
print(repr(book))

Python Patterns (240 pages)
BookDescription('Python Patterns', 240)


## 13. Operator overloading

Special methods let custom objects participate in familiar operations. For example, `left + right` can call `left.__add__(right)`. Handle unsupported operand types deliberately; returning `NotImplemented` lets Python try the other operand's reflected operation or raise a useful `TypeError`.

In [16]:
class PageCount:
    def __init__(self, pages):
        self.pages = pages

    def __add__(self, other):
        if not isinstance(other, PageCount):
            return NotImplemented
        return PageCount(self.pages + other.pages)

    def __repr__(self):
        return f"PageCount({self.pages})"

first_book = PageCount(120)
second_book = PageCount(180)
print(first_book + second_book)

PageCount(300)


Other common special methods include `__len__` for `len(object)`, `__eq__` for `==`, and `__mul__` for `*`. Define only behavior that is intuitive for your type.

## 14. Equality (`==`) versus identity (`is`)

`==` asks whether two values are equal according to their type's equality behavior. `is` asks whether two references point to the exact same object. If value-based equality is desired for a custom class, implement `__eq__`. Use `is` especially for singleton checks such as `value is None`.

In [17]:
class NamedValue:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        if not isinstance(other, NamedValue):
            return NotImplemented
        return self.value == other.value

value_one = NamedValue("same")
value_two = NamedValue("same")
alias = value_one
print(value_one == value_two)  # Same value.
print(value_one is value_two)  # Different objects.
print(value_one is alias)      # Same object.

True
False
True


## 15. References, shallow copies, and deep copies

Assignment creates another reference; it does not copy an object. A **shallow copy** creates a new outer object but keeps references to nested objects. A **deep copy** recursively copies nested objects too. Choose based on whether nested mutable state should remain shared.

In [18]:
import copy

class ProjectMember:
    def __init__(self, name, skills):
        self.name = name
        self.skills = skills

original = ProjectMember("Amit", ["Python", "SQL"])
alias = original
shallow = copy.copy(original)
deep = copy.deepcopy(original)

shallow.name = "Rahul"
shallow.skills.append("AWS")  # The skills list is still shared with original.
deep.skills.append("Testing")  # The deep copy has its own skills list.

print(original.name, original.skills)
print(alias is original)
print(shallow.name, shallow.skills)
print(deep.name, deep.skills)

Amit ['Python', 'SQL', 'AWS']
True
Rahul ['Python', 'SQL', 'AWS']
Amit ['Python', 'SQL', 'Testing']


## 16. Integrated example: a small shape model

This example combines abstraction, inheritance, overriding, validation, and polymorphism. The caller can ask every shape for its area without knowing which concrete shape it received.

In [19]:
import math
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        """Return the area of this shape."""
        raise NotImplementedError

class Circle(Shape):
    def __init__(self, radius):
        if radius < 0:
            raise ValueError("Radius cannot be negative")
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, width, height):
        if width < 0 or height < 0:
            raise ValueError("Dimensions cannot be negative")
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

shapes = [Circle(5), Rectangle(4, 6)]
for shape in shapes:
    print(f"{type(shape).__name__}: area = {shape.area():.2f}")

Circle: area = 78.54
Rectangle: area = 24.00


## 17. Practice exercises

Try each exercise in a new code cell. Use the earlier examples as reference, but write the solution before looking anything up.

1. Create a `Car` class with `brand` and `model`, plus a method that returns a readable description. Create two cars.
2. Create a `LibraryBook` class with a class-level `total_created` counter and per-instance title and author. Increment the counter for each new book.
3. Add a `@property` setter to a class that rejects a value outside a sensible range. Demonstrate catching the error.
4. Define an abstract `Notification` class with `send(message)`, then implement email and text notifications.
5. Make a `Car` contain an interchangeable engine object. Demonstrate two engine implementations.
6. Implement `__eq__` for a small value object and explain the difference between equality and identity.
7. Create two objects with a nested list, then compare assignment, `copy.copy`, and `copy.deepcopy`.

## Quick recap

- Classes define behavior and data; instances hold per-object state.
- Keep mutable per-instance data in `__init__`, not on the class.
- Use inheritance for genuine is-a relationships and composition for has-a relationships.
- Properties, methods, and ABCs make object interfaces easier to use correctly.
- `super()` follows the MRO; `==` compares values and `is` compares identity.
- Assignment shares an object; shallow and deep copies differ in their handling of nested objects.